<a href="https://colab.research.google.com/github/Daprosero/Domain_Adaptation/blob/main/CREDA/Notebooks/Tables_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import json
from importlib import metadata as importlib_metadata
import os
import re
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

MIGRATED_PROJECT_MARKER = ("src", "CREDA")
DEFAULT_DOMAIN_ADAPTATION_REPOSITORY = 'https://github.com/Daprosero/Domain_Adaptation.git'
DOMAIN_ADAPTATION_REPOSITORY = os.environ.get('DOMAIN_ADAPTATION_REPOSITORY', DEFAULT_DOMAIN_ADAPTATION_REPOSITORY)
OFFICECALTECH_REPOSITORY = "https://github.com/ZhangJUJU/OfficeCaltechDomainAdaptation.git"
IMAGECLEF_FILE_ID = "1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf"


def run_command(*command):
    return subprocess.run(command, check=True)


def detect_runtime():
    if "COLAB_RELEASE_TAG" in os.environ:
        return "Colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "Kaggle"
    return "local"


RUNTIME = detect_runtime()
MANAGED_CLOUD_REQUIREMENTS = {"torch", "torchvision", "jupyter", "ipykernel"}
BINARY_EXTENSION_SUFFIXES = (".so", ".pyd", ".dll", ".dylib")


def requirement_name(requirement_line):
    requirement = requirement_line.split("#", 1)[0].split(";", 1)[0].strip()
    match = re.match(r"[A-Za-z0-9_.-]+", requirement)
    return match.group(0).lower().replace("_", "-") if match else None


def write_cloud_requirements(requirements_path, destination):
    cloud_requirements = [
        line for line in requirements_path.read_text(encoding="utf-8").splitlines(keepends=True)
        if requirement_name(line) not in MANAGED_CLOUD_REQUIREMENTS
    ]
    destination.write_text("".join(cloud_requirements), encoding="utf-8")


def binary_packages_from_report(report_path):
    report = json.loads(report_path.read_text(encoding="utf-8"))
    changed_binary_packages = []
    for installation in report.get("install", []):
        package_name = installation.get("metadata", {}).get("name")
        if not package_name:
            continue
        try:
            installed_files = importlib_metadata.distribution(package_name).files
        except importlib_metadata.PackageNotFoundError:
            changed_binary_packages.append(package_name)
            continue
        if installed_files is None or any(
            str(path).lower().endswith(BINARY_EXTENSION_SUFFIXES) for path in installed_files
        ):
            changed_binary_packages.append(package_name)
    return changed_binary_packages


def request_cloud_restart(changed_binary_packages):
    package_list = ", ".join(changed_binary_packages)
    if RUNTIME == "Kaggle":
        raise RuntimeError(
            f"Kaggle installed binary package(s): {package_list}. Restart the session and rerun this notebook before scientific imports."
        )

    print(f"Colab installed binary package(s): {package_list}. Requesting a runtime restart before scientific imports.")
    try:
        from google.colab import runtime as colab_runtime
        restart = getattr(colab_runtime, "restart", None)
        if callable(restart):
            restart()
        else:
            from IPython import get_ipython
            shell = get_ipython()
            if shell is not None and getattr(shell, "kernel", None) is not None:
                shell.kernel.do_shutdown(restart=True)
    except (ImportError, AttributeError):
        pass
    raise RuntimeError(
        "Colab dependencies changed. Restart the runtime if it did not restart automatically, then rerun this notebook."
    )


def install_project_requirements(requirements_path):
    if RUNTIME == "local":
        run_command(sys.executable, "-m", "pip", "install", "-r", str(requirements_path))
        return

    with tempfile.TemporaryDirectory(prefix="domain-adaptation-pip-") as temporary_directory:
        temporary_directory = Path(temporary_directory)
        cloud_requirements_path = temporary_directory / "requirements-cloud.txt"
        report_path = temporary_directory / "pip-install-report.json"
        write_cloud_requirements(requirements_path, cloud_requirements_path)
        run_command(
            sys.executable, "-m", "pip", "install", "--report", str(report_path),
            "-r", str(cloud_requirements_path),
        )
        if not report_path.is_file():
            raise RuntimeError("pip did not produce an installation report; restart before scientific imports.")
        changed_binary_packages = binary_packages_from_report(report_path)
        if changed_binary_packages:
            request_cloud_restart(changed_binary_packages)


def is_repository_root(candidate):
    return candidate.joinpath(*MIGRATED_PROJECT_MARKER).is_dir() and (candidate / 'pyproject.toml').is_file()


def cloud_repository_destination():
    if 'COLAB_RELEASE_TAG' in os.environ:
        return Path('/content/Domain_Adaptation')
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return Path('/kaggle/working/Domain_Adaptation')
    return None


def find_repository_root():
    candidates = [Path(os.environ["DOMAIN_ADAPTATION_ROOT"])] if "DOMAIN_ADAPTATION_ROOT" in os.environ else []
    candidates.extend([Path.cwd(), *Path.cwd().parents, Path("/content/Domain_Adaptation"), Path("/kaggle/working/Domain_Adaptation"), Path("/kaggle/input/domain-adaptation")])
    for candidate in candidates:
        if (candidate / "src" / "CREDA").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate.resolve()

    destination = cloud_repository_destination()
    if destination is not None:
        if not destination.exists():
            run_command('git', 'clone', DOMAIN_ADAPTATION_REPOSITORY, str(destination))
        if is_repository_root(destination):
            return destination.resolve()
        raise FileNotFoundError(f'Cloud repository destination is not a valid CREDA checkout: {destination}')

    cache_root = Path(os.environ.get("DOMAIN_ADAPTATION_CACHE", Path.cwd() / ".domain-adaptation-cache")).expanduser()
    raise FileNotFoundError(
        "A local migrated Domain_Adaptation checkout containing src/CREDA is required. "
        "Set DOMAIN_ADAPTATION_ROOT to that checkout."
    )


REPOSITORY_ROOT = find_repository_root()
CACHE_ROOT = Path(os.environ.get(
    "DOMAIN_ADAPTATION_CACHE",
    REPOSITORY_ROOT.parent if REPOSITORY_ROOT.parent.name == ".domain-adaptation-cache" else REPOSITORY_ROOT / ".domain-adaptation-cache",
)).expanduser().resolve()
OFFICECALTECH_ROOT = CACHE_ROOT / "OfficeCaltechDomainAdaptation" / "images"
IMAGECLEF_ROOT = CACHE_ROOT / "ImageCLEF-DA" / "image_CLEF"

if not (CACHE_ROOT / "OfficeCaltechDomainAdaptation").exists():
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    run_command("git", "clone", OFFICECALTECH_REPOSITORY, str(CACHE_ROOT / "OfficeCaltechDomainAdaptation"))

requirements_path = REPOSITORY_ROOT / "requirements.txt"
if not requirements_path.is_file():
    raise FileNotFoundError(f"Canonical requirements file is missing: {requirements_path}")
install_project_requirements(requirements_path)
imageclef_zip = CACHE_ROOT / "ImageCLEF-DA.zip"
if not imageclef_zip.is_file():
    run_command("gdown", "--id", IMAGECLEF_FILE_ID, "--output", str(imageclef_zip))
if not IMAGECLEF_ROOT.is_dir():
    with zipfile.ZipFile(imageclef_zip) as archive:
        archive.extractall(CACHE_ROOT / "ImageCLEF-DA")

os.environ.setdefault("DOMAIN_ADAPTATION_ROOT", str(REPOSITORY_ROOT))
os.environ.setdefault("DOMAIN_ADAPTATION_CACHE", str(CACHE_ROOT))
os.environ.setdefault("DOMAIN_ADAPTATION_IMAGECLEF_ROOT", str(IMAGECLEF_ROOT))
os.environ.setdefault("DOMAIN_ADAPTATION_OFFICECALTECH_ROOT", str(OFFICECALTECH_ROOT))
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from CREDA.artifacts import (
    get_latex_rows,
    load_dataset_results,
    warn_on_incomplete_table_results,
    resolve_artifact_paths,
    resolve_imageclef_root,
    resolve_officecaltech_root,
)
ARTIFACT_PATHS = resolve_artifact_paths(REPOSITORY_ROOT)
IMAGECLEF_ROOT = resolve_imageclef_root(REPOSITORY_ROOT)
OFFICECALTECH_ROOT = resolve_officecaltech_root(REPOSITORY_ROOT)


  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 1.0 MB/s  0:00:11 eta 0:00:01
Using cached pytz-2026.3.post1-py2.py3-none-any.whl (508 kB)
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.5
    Uninstalling pandas-3.0.5:0m╺━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]
      Successfully uninstalled pandas-3.0.5━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [9]:
Digi18 = load_dataset_results("resnet18", "MNIST-USPS-SVHN")
Digi50 = load_dataset_results("resnet50", "MNIST-USPS-SVHN")
DigiTr = load_dataset_results("vit_tiny_patch16_224", "MNIST-USPS-SVHN")

In [10]:
methods = ['Baseline', 'DANN', 'ADDA', 'CDAN', 'CREDA']
transfers = [('M', 'U'), ('M', 'S'), ('U', 'M'), ('U', 'S'), ('S', 'M'), ('S', 'U')]

warn_on_incomplete_table_results(Digi18, methods, transfers)
warn_on_incomplete_table_results(Digi50, methods, transfers)
warn_on_incomplete_table_results(DigiTr, methods, transfers)

rows_18 = get_latex_rows(Digi18, "ResNet-18", methods, transfers)
rows_50 = get_latex_rows(Digi50, "ResNet-50", methods, transfers)
rows_vit = get_latex_rows(DigiTr, "ViT-Tiny", methods, transfers)

all_rows = rows_18 + rows_50 + rows_vit

latex = []
latex.append("\\begin{table}[H]")
latex.append("\\centering")
latex.append("\\caption{Accuracy (\\%) on Digits for unsupervised domain adaptation using different backbone architectures.}")
latex.append("\\resizebox{0.98\\linewidth}{!}{")
latex.append("\\begin{tabular}{@{}lcccccccc@{}}")
latex.append("\\toprule")
latex.append("\\textbf{Model} & \\textbf{Method} & M $\\rightarrow$ U & M $\\rightarrow$ S & U $\\rightarrow$ M & U $\\rightarrow$ S & S $\\rightarrow$ M & S $\\rightarrow$ U & \\textbf{Avg} \\\\")
latex.append("\\midrule")

# Generación con \multirow
current_model = None
for i, (model, method, values) in enumerate(all_rows):
    if model != current_model:
        latex.append(f"\\multirow{{5}}{{*}}{{{model}}} & {method} & " + " & ".join(values) + " \\\\")
        current_model = model
    else:
        latex.append(f" & {method} & " + " & ".join(values) + " \\\\")

latex.append("\\bottomrule")
latex.append("\\end{tabular}")
latex.append("}")
latex.append("\\label{tab:digits}")
latex.append("\\end{table}")
# Resultado final
latex_code = "\n".join(latex)
print(latex_code)

\begin{table}[H]
\centering
\caption{Accuracy (\%) on Digits for unsupervised domain adaptation using different backbone architectures.}
\resizebox{0.98\linewidth}{!}{
\begin{tabular}{@{}lcccccccc@{}}
\toprule
\textbf{Model} & \textbf{Method} & M $\rightarrow$ U & M $\rightarrow$ S & U $\rightarrow$ M & U $\rightarrow$ S & S $\rightarrow$ M & S $\rightarrow$ U & \textbf{Avg} \\
\midrule
\multirow{5}{*}{ResNet-18} & Baseline & 56.73~$\pm$~20.93 & 22.04~$\pm$~14.77 & 76.64~$\pm$~15.49 & 9.68~$\pm$~10.60 & 69.49~$\pm$~16.82 & 74.69~$\pm$~15.88 & 51.55~$\pm$~15.75 \\
 & DANN & 86.20~$\pm$~10.64 & 19.28~$\pm$~13.61 & 80.84~$\pm$~13.65 & 28.65~$\pm$~16.12 & 72.64~$\pm$~15.72 & 70.66~$\pm$~16.33 & 59.71~$\pm$~14.35 \\
 & ADDA & 7.68~$\pm$~9.13 & 30.99~$\pm$~16.93 & 83.30~$\pm$~12.67 & 28.27~$\pm$~16.22 & 74.32~$\pm$~15.31 & 66.27~$\pm$~15.85 & 48.47~$\pm$~14.35 \\
 & CDAN & 81.99~$\pm$~11.92 & 15.08~$\pm$~12.80 & 25.12~$\pm$~15.75 & 14.19~$\pm$~12.72 & 56.42~$\pm$~17.01 & 66.45~$\pm$~17.51 & 

In [11]:
IRes18 = load_dataset_results("resnet18", "ImageCLEF")
IRes50 = load_dataset_results("resnet50", "ImageCLEF")
ITrans = load_dataset_results("vit_tiny_patch16_224", "ImageCLEF")

In [12]:
methods = ['Baseline', 'DANN', 'ADDA', 'CDAN', 'CREDA']
transfers = [('I', 'P'), ('I', 'C'), ('P', 'I'), ('P', 'C'), ('C', 'I'), ('C', 'P')]
warn_on_incomplete_table_results(IRes18, methods, transfers)
warn_on_incomplete_table_results(IRes50, methods, transfers)
warn_on_incomplete_table_results(ITrans, methods, transfers)
rows_18 = get_latex_rows(IRes18, "ResNet-18", methods, transfers)
rows_50 = get_latex_rows(IRes50, "ResNet-50", methods, transfers)
rows_vit = get_latex_rows(ITrans, "ViT-Tiny", methods, transfers)
all_rows = rows_18 + rows_50 + rows_vit
latex = []
latex.append("\\begin{table}[H]")
latex.append("\\centering")
latex.append("\\caption{Accuracy (\\%) on ImageCLEF-DA for unsupervised domain adaptation using different backbone architectures.}")
latex.append("\\resizebox{0.98\\linewidth}{!}{")
latex.append("\\begin{tabular}{@{}lcccccccc@{}}")
latex.append("\\toprule")
latex.append("\\textbf{Model} & \\textbf{Method} & I $\\rightarrow$ P & I $\\rightarrow$ C & P $\\rightarrow$ I & P $\\rightarrow$ C & C $\\rightarrow$ I & C $\\rightarrow$ P & \\textbf{Avg} \\\\")
latex.append("\\midrule")
current_model = None
for i, (model, method, values) in enumerate(all_rows):
    if model != current_model:
        latex.append(f"\\multirow{{5}}{{*}}{{{model}}} & {method} & " + " & ".join(values) + " \\\\")
        current_model = model
    else:
        latex.append(f" & {method} & " + " & ".join(values) + " \\\\")
latex.append("\\bottomrule")
latex.append("\\end{tabular}")
latex.append("}")
latex.append("\\label{tab:imageclef}")
latex.append("\\end{table}")
latex_code = "\n".join(latex)
print(latex_code)

\begin{table}[H]
\centering
\caption{Accuracy (\%) on ImageCLEF-DA for unsupervised domain adaptation using different backbone architectures.}
\resizebox{0.98\linewidth}{!}{
\begin{tabular}{@{}lcccccccc@{}}
\toprule
\textbf{Model} & \textbf{Method} & I $\rightarrow$ P & I $\rightarrow$ C & P $\rightarrow$ I & P $\rightarrow$ C & C $\rightarrow$ I & C $\rightarrow$ P & \textbf{Avg} \\
\midrule
\multirow{5}{*}{ResNet-18} & Baseline & 58.00~$\pm$~21.71 & 76.83~$\pm$~21.52 & 68.00~$\pm$~19.30 & 76.50~$\pm$~19.05 & 49.83~$\pm$~24.44 & 38.67~$\pm$~25.60 & 61.31~$\pm$~21.94 \\
 & DANN & 60.00~$\pm$~25.00 & 85.56~$\pm$~13.55 & 66.67~$\pm$~16.43 & 78.89~$\pm$~18.04 & 76.67~$\pm$~17.78 & 57.78~$\pm$~23.74 & 70.93~$\pm$~19.09 \\
 & ADDA & 68.89~$\pm$~20.87 & 77.78~$\pm$~11.10 & 71.11~$\pm$~19.09 & 81.11~$\pm$~16.39 & 74.44~$\pm$~14.56 & 58.89~$\pm$~21.62 & 72.04~$\pm$~17.27 \\
 & CDAN & 56.67~$\pm$~23.31 & 62.22~$\pm$~15.50 & 65.56~$\pm$~16.71 & 58.89~$\pm$~16.28 & 68.89~$\pm$~21.62 & 47.78~$\pm$

In [13]:
ORes18 = load_dataset_results("resnet18", "Office-Caltech")
ORes50 = load_dataset_results("resnet50", "Office-Caltech")
OTrans = load_dataset_results("vit_tiny_patch16_224", "Office-Caltech")

In [14]:
methods = ['Baseline', 'DANN', 'ADDA', 'CDAN', 'CREDA']
transfers = [('A', 'W'), ('A', 'D'), ('W', 'A'), ('W', 'D'), ('D', 'A'), ('D', 'W')]
warn_on_incomplete_table_results(ORes18, methods, transfers)
warn_on_incomplete_table_results(ORes50, methods, transfers)
warn_on_incomplete_table_results(OTrans, methods, transfers)
rows_18 = get_latex_rows(ORes18, "ResNet-18", methods, transfers)
rows_50 = get_latex_rows(ORes50, "ResNet-50", methods, transfers)
rows_vit = get_latex_rows(OTrans, "ViT-Tiny", methods, transfers)
all_rows = rows_18 + rows_50 + rows_vit
latex = []
latex.append("\\begin{table}[H]")
latex.append("\\centering")
latex.append("\\caption{Accuracy (\\%) on Office-31 for unsupervised domain adaptation using different backbone architectures.}")
latex.append("\\resizebox{0.98\\linewidth}{!}{")
latex.append("\\begin{tabular}{@{}lcccccccc@{}}")
latex.append("\\toprule")
latex.append("\\textbf{Model} & \\textbf{Method} & A $\\rightarrow$ W & A $\\rightarrow$ D & W $\\rightarrow$ A & W $\\rightarrow$ D & D $\\rightarrow$ A & D $\\rightarrow$ W & \\textbf{Avg} \\\\")
latex.append("\\midrule")
current_model = None
for i, (model, method, values) in enumerate(all_rows):
    if model != current_model:
        latex.append(f"\\multirow{{5}}{{*}}{{{model}}} & {method} & " + " & ".join(values) + " \\\\")
        current_model = model
    else:
        latex.append(f" & {method} & " + " & ".join(values) + " \\\\")
latex.append("\\bottomrule")
latex.append("\\end{tabular}")
latex.append("}")
latex.append("\\label{tab:office31}")
latex.append("\\end{table}")
latex_code = "\n".join(latex)
print(latex_code)

\begin{table}[H]
\centering
\caption{Accuracy (\%) on Office-31 for unsupervised domain adaptation using different backbone architectures.}
\resizebox{0.98\linewidth}{!}{
\begin{tabular}{@{}lcccccccc@{}}
\toprule
\textbf{Model} & \textbf{Method} & A $\rightarrow$ W & A $\rightarrow$ D & W $\rightarrow$ A & W $\rightarrow$ D & D $\rightarrow$ A & D $\rightarrow$ W & \textbf{Avg} \\
\midrule
\multirow{5}{*}{ResNet-18} & Baseline & 50.51~$\pm$~29.45 & 55.41~$\pm$~25.37 & 54.91~$\pm$~34.50 & 96.82~$\pm$~7.98 & 46.56~$\pm$~34.31 & 78.98~$\pm$~28.90 & 63.86~$\pm$~26.75 \\
 & DANN & 73.33~$\pm$~15.81 & 87.50~$\pm$~12.50 & 67.36~$\pm$~16.68 & 100.00~$\pm$~0.00 & 51.39~$\pm$~17.09 & 84.44~$\pm$~12.29 & 77.34~$\pm$~12.40 \\
 & ADDA & 62.22~$\pm$~26.71 & 87.50~$\pm$~12.50 & 54.17~$\pm$~19.65 & 100.00~$\pm$~0.00 & 59.72~$\pm$~17.45 & 84.44~$\pm$~9.41 & 74.68~$\pm$~14.29 \\
 & CDAN & 64.44~$\pm$~14.72 & 75.00~$\pm$~12.50 & 50.69~$\pm$~21.64 & 100.00~$\pm$~0.00 & 51.39~$\pm$~16.54 & 88.89~$\pm$~12.2